In [10]:
import pandas as pd
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import numpy as np
import joblib
import json
import os
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
benign_path = '/content/drive/MyDrive/SIH_dataset/l2-benign.csv'
malicious_path = '/content/drive/MyDrive/SIH_dataset/l2-malicious.csv'

In [3]:
benign = pd.read_csv(benign_path).assign(label=0)
malicious = pd.read_csv(malicious_path).assign(label=1)

In [4]:
malicious_sampled = malicious.sample(n=len(benign), random_state=42)
df = pd.concat([benign, malicious_sampled], ignore_index=True)

In [5]:
drop_cols = ['SourceIP', 'DestinationIP', 'SourcePort', 'DestinationPort', 'TimeStamp']
df = df.drop(columns=[col for col in drop_cols if col in df.columns])


In [6]:

print("Prototype dataset ready. Shape:", df.shape)
print(df['label'].value_counts())

Prototype dataset ready. Shape: (39614, 31)
label
0    19807
1    19807
Name: count, dtype: int64


In [7]:
df.head()

,Duration,FlowBytesSent,FlowSentRate,FlowBytesReceived,FlowReceivedRate,PacketLengthVariance,PacketLengthStandardDeviation,PacketLengthMean,PacketLengthMedian,PacketLengthMode,...,ResponseTimeTimeVariance,ResponseTimeTimeStandardDeviation,ResponseTimeTimeMean,ResponseTimeTimeMedian,ResponseTimeTimeMode,ResponseTimeTimeSkewFromMedian,ResponseTimeTimeSkewFromMode,ResponseTimeTimeCoefficientofVariation,Label,label
0,95.081550,62311,655.342703,65358,687.388878,7474.676771,86.456213,135.673751,102.0,54,...,0.001053,0.032457,0.027624,0.026854,0.026822,0.071187,0.024715,1.174948,Benign,0
1,122.309318,93828,767.136973,101232,827.672018,10458.118598,102.264943,141.245474,114.0,54,...,0.001170,0.034200,0.024387,0.021043,0.026981,0.293297,-0.075845,1.402382,Benign,0
2,120.958413,38784,320.639127,38236,316.108645,7300.293933,85.441758,133.715278,89.0,54,...,0.000785,0.028021,0.029238,0.026922,0.026855,0.248064,0.085061,0.958348,Benign,0
3,110.501080,61993,561.017141,69757,631.278898,8499.282518,92.191553,139.123548,114.0,54,...,0.000411,0.020274,0.019925,0.019268,0.026918,0.097199,-0.344926,1.017535,Benign,0
4,54.229891,83641,1542.341289,76804,1416.266907,8052.745751,89.737092,138.913420,114.0,114,...,0.079079,0.281209,0.025930,0.000046,0.000021,0.276133,0.092135,10.844829,Benign,0


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39614 entries, 0 to 39613
Data columns (total 31 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   Duration                                39614 non-null  float64
 1   FlowBytesSent                           39614 non-null  int64  
 2   FlowSentRate                            39614 non-null  float64
 3   FlowBytesReceived                       39614 non-null  int64  
 4   FlowReceivedRate                        39614 non-null  float64
 5   PacketLengthVariance                    39614 non-null  float64
 6   PacketLengthStandardDeviation           39614 non-null  float64
 7   PacketLengthMean                        39614 non-null  float64
 8   PacketLengthMedian                      39614 non-null  float64
 9   PacketLengthMode                        39614 non-null  int64  
 10  PacketLengthSkewFromMedian              39614 non-null  fl

In [9]:
if 'Label' in df.columns:
    df = df.drop(columns=['Label'])
df = df.fillna(0)
X = df.drop(columns=['label'])
y = df['label']
print(f"Features matrix shape (X): {X.shape}")
print(f"Target vector shape (y):   {y.shape}")

Features matrix shape (X): (39614, 29)
Target vector shape (y):   (39614,)


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training samples: {X_train.shape[0]:,} | Testing samples: {X_test.shape[0]:,}")
rf_dns = RandomForestClassifier(
    n_estimators=150,
    max_depth=16,
    random_state=42,
    n_jobs=-1
)
print("\nTraining Random Forest baseline on DoH statistical features...")
rf_dns.fit(X_train, y_train)
print("Training complete!")
y_probs = rf_dns.predict_proba(X_test)[:, 1]

Training samples: 31,691 | Testing samples: 7,923

Training Random Forest baseline on DoH statistical features...
Training complete!


In [13]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training samples: {X_train.shape[0]:,} | Testing samples: {X_test.shape[0]:,}")
print("Training class distribution:\n", y_train.value_counts())

Training samples: 31,691 | Testing samples: 7,923
Training class distribution:
 label
1    15846
0    15845
Name: count, dtype: int64


In [14]:
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
xgb_dns = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    tree_method='hist',
    random_state=42,
    n_jobs=-1
)
print("Training XGBoost Classifier...")
xgb_dns.fit(X_train, y_train)
rf_dns = RandomForestClassifier(
    n_estimators=150,
    max_depth=16,
    random_state=42,
    n_jobs=-1
)
print("Training Random Forest Classifier...")
rf_dns.fit(X_train, y_train)
print("Training complete!")

Training XGBoost Classifier...
Training Random Forest Classifier...
Training complete!


In [15]:
import time
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
def evaluate_detector(model, name, X_test, y_test, threshold=0.50):
    start_time = time.perf_counter()
    y_probs = model.predict_proba(X_test)[:, 1]
    total_time = time.perf_counter() - start_time
    latency_per_flow_us = (total_time / len(X_test)) * 1_000_000
    y_preds = (y_probs >= threshold).astype(int)
    cm = confusion_matrix(y_test, y_preds)
    tn, fp, fn, tp = cm.ravel()
    fpr = (fp / (fp + tn)) * 100 if (fp + tn) > 0 else 0.0
    recall = (tp / (tp + fn)) * 100 if (tp + fn) > 0 else 0.0
    auc = roc_auc_score(y_test, y_probs)

    print("=" * 65)
    print(f"{name.upper()} EVALUATION (Threshold = {threshold:.2f})")
    print("=" * 65)
    print(classification_report(y_test, y_preds, target_names=['Benign DoH (0)', 'Malicious Tunnel (1)'], digits=4))
    print(f"ROC-AUC Score:          {auc:.4f}")
    print(f"Malicious Recall:       {recall:.2f}%")
    print(f"False Positive Rate:    {fpr:.2f}%")
    print(f"Inference Latency/Flow: {latency_per_flow_us:.2f} µs\n")

    cm_df = pd.DataFrame(
        cm,
        index=['Actual Benign (0)', 'Actual Malicious (1)'],
        columns=['Predicted Benign (0)', 'Predicted Malicious (1)']
    )
    display(cm_df)
    return y_probs
xgb_probs = evaluate_detector(xgb_dns, "XGBoost", X_test, y_test)
rf_probs = evaluate_detector(rf_dns, "Random Forest", X_test, y_test)

XGBOOST EVALUATION (Threshold = 0.50)
                      precision    recall  f1-score   support

      Benign DoH (0)     0.9997    1.0000    0.9999      3962
Malicious Tunnel (1)     1.0000    0.9997    0.9999      3961

            accuracy                         0.9999      7923
           macro avg     0.9999    0.9999    0.9999      7923
        weighted avg     0.9999    0.9999    0.9999      7923

ROC-AUC Score:          1.0000
Malicious Recall:       99.97%
False Positive Rate:    0.00%
Inference Latency/Flow: 4.14 µs



,Predicted Benign (0),Predicted Malicious (1)
Actual Benign (0),3962,0
Actual Malicious (1),1,3960


RANDOM FOREST EVALUATION (Threshold = 0.50)
                      precision    recall  f1-score   support

      Benign DoH (0)     0.9987    1.0000    0.9994      3962
Malicious Tunnel (1)     1.0000    0.9987    0.9994      3961

            accuracy                         0.9994      7923
           macro avg     0.9994    0.9994    0.9994      7923
        weighted avg     0.9994    0.9994    0.9994      7923

ROC-AUC Score:          1.0000
Malicious Recall:       99.87%
False Positive Rate:    0.00%
Inference Latency/Flow: 11.92 µs



,Predicted Benign (0),Predicted Malicious (1)
Actual Benign (0),3962,0
Actual Malicious (1),5,3956


In [16]:

drive_dir = "/content/drive/MyDrive/SIH_dataset/"
os.makedirs(drive_dir, exist_ok=True)
output_model_file = os.path.join(drive_dir, "dns_tunnelling_xgb_model.pkl")
feature_list_file = os.path.join(drive_dir, "dns_tunnelling_schema.json")
joblib.dump(xgb_dns, output_model_file)
print(f"Model permanently saved to Drive: {output_model_file}")
model_metadata = {
    "model_name": "dns_tunnel_xgb",
    "version": "1.0",
    "dataset": "CIC-DoHBrw-2020 Layer 2",
    "threat_class": "DNS_TUNNELLING",
    "classification_threshold": 0.50,
    "expected_features": list(X_train.columns)
}
with open(feature_list_file, "w") as f:
    json.dump(model_metadata, f, indent=4)
print(f"Metadata permanently saved to Drive: {feature_list_file}")

Model permanently saved to Drive: /content/drive/MyDrive/SIH_dataset/dns_tunnelling_xgb_model.pkl
Metadata permanently saved to Drive: /content/drive/MyDrive/SIH_dataset/dns_tunnelling_schema.json
